In [5]:
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

# ---------------------------------------------------------------------
# Global random map for Rough Terrain
# ---------------------------------------------------------------------
np.random.seed(0)  # So that changing sliders won't completely re-randomize
global_random_map = np.random.rand(512, 512)  # Pre-generate at max size

# ---------------------------------------------------------------------
# Core heightmap generator
# ---------------------------------------------------------------------
def generate_heightmap(size, n_regions, region_types, region_params):
    """
    Generate a heightmap of shape (size, size) split into a grid of n_regions,
    where n_regions ∈ {1, 4, 9}.
    
    - region_types, region_params are lists of length n_regions, each describing
      that region's type (Staircase or Rough Terrain) and parameter (amplitude).
    """
    heightmap = np.zeros((size, size), dtype=np.float32)
    
    # Figure out the grid layout
    # 1 -> (1,1), 4 -> (2,2), 9 -> (3,3)
    grid_size = int(np.sqrt(n_regions))  # 1, 2, or 3
    sub_size = size // grid_size  # how large each sub-region is

    # For "Staircase":
    num_stairs = 10  # number of steps in horizontal direction

    # Go region by region
    for idx in range(n_regions):
        r_type = region_types[idx]
        param = region_params[idx]
        
        # Which row/column in the grid?
        row = idx // grid_size
        col = idx % grid_size
        
        # Region bounding box
        y_start = row * sub_size
        y_end   = (row + 1) * sub_size if row < grid_size - 1 else size
        x_start = col * sub_size
        x_end   = (col + 1) * sub_size if col < grid_size - 1 else size
        
        if r_type == "Staircase":
            # Horizontal staircase across x
            region_width = x_end - x_start
            step_size = region_width / float(num_stairs)

            for x in range(x_start, x_end):
                step_index = int((x - x_start) // step_size)
                # clamp at boundary
                if step_index >= num_stairs:
                    step_index = num_stairs - 1
                stair_height = (step_index / (num_stairs - 1)) * param
                heightmap[y_start:y_end, x] = stair_height

        elif r_type == "Rough Terrain":
            # Sliced from the top-left portion of our global random map
            # Scale by param
            rough_chunk = global_random_map[y_start:y_end, x_start:x_end]
            heightmap[y_start:y_end, x_start:x_end] = rough_chunk * param

        else:
            # Default to zero if something else
            heightmap[y_start:y_end, x_start:x_end] = 0.0

    return heightmap


# ---------------------------------------------------------------------
# Interactive widgets
# ---------------------------------------------------------------------

# 1) Size slider
size_slider = widgets.IntSlider(
    value=256, min=16, max=512, step=16,
    description='Size', continuous_update=False
)

# 2) Number of regions: only allow 1,4,9
n_regions_dropdown = widgets.Dropdown(
    options=[1, 4, 9],
    value=1,  # default
    description='Regions'
)

# We will create up to 9 region controls (type + param),
# and we will hide or show them depending on the chosen n_regions.
region_type_widgets = []
region_param_widgets = []

for i in range(9):
    # A dropdown for region type
    rt = widgets.Dropdown(
        options=["Staircase", "Rough Terrain"],
        value="Staircase",
        description=f"R{i+1} Type"
    )
    # A slider for region param
    rp = widgets.FloatSlider(
        value=0.3, min=0.0, max=1.0, step=0.05,
        description=f"Param {i+1}"
    )
    region_type_widgets.append(rt)
    region_param_widgets.append(rp)

# Function to hide/show the appropriate region widgets
def update_region_widget_visibility(n):
    for i in range(9):
        if i < n:
            region_type_widgets[i].layout.display = 'block'
            region_param_widgets[i].layout.display = 'block'
        else:
            region_type_widgets[i].layout.display = 'none'
            region_param_widgets[i].layout.display = 'none'

# Initially set to 1 region
update_region_widget_visibility(n_regions_dropdown.value)

# We observe changes in the dropdown to hide/show region widgets
def on_n_regions_change(change):
    if change['name'] == 'value':
        update_region_widget_visibility(change['new'])

n_regions_dropdown.observe(on_n_regions_change, names='value')

# 3) A button to "Update" the plot or you can use auto-updating
update_button = widgets.Button(description="Update Heightmap")


# 4) An output area to display the resulting image
output = widgets.Output()

# Callback to regenerate and display the heightmap
def update_plot(_=None):
    with output:
        output.clear_output()
        
        # Gather input values
        size = size_slider.value
        n_regions = n_regions_dropdown.value
        
        # We'll take the first n_regions from the lists
        chosen_types = [w.value for w in region_type_widgets[:n_regions]]
        chosen_params = [w.value for w in region_param_widgets[:n_regions]]

        # Generate the heightmap
        hm = generate_heightmap(size, n_regions, chosen_types, chosen_params)
        
        # Show it
        plt.figure(figsize=(6,6))
        plt.title(f"Heightmap ({n_regions} region{'s' if n_regions>1 else ''})")
        plt.imshow(hm, cmap='gray', origin='lower')
        plt.colorbar(label='Height')
        plt.show()

# Wire the button to the callback
update_button.on_click(update_plot)

# If you want immediate re-draw when changing sliders (instead of pressing the button),
# you can observe them as well:
size_slider.observe(update_plot, names='value')
for w in region_type_widgets + region_param_widgets:
    w.observe(update_plot, names='value')


# ---------------------------------------------------------------------
# Display the UI
# ---------------------------------------------------------------------
control_widgets = [size_slider, n_regions_dropdown] 
for i in range(9):
    control_widgets.append(region_type_widgets[i])
    control_widgets.append(region_param_widgets[i])
control_widgets.append(update_button)

ui = widgets.VBox(control_widgets)

display(ui, output)



Output()